# WAIND GPT 0.7 — preparação e avaliação de estimativas de intensidade

Intensidade humana primeiro. Este notebook estima a percepção de 1 a 5 em relatos de odor; não detecta presença de cheiro, não mede substâncias e não altera relatos/mapa.

**Padrão: modo técnico com lote sintético. Seus resultados verificam somente o pipeline. Não são resultados de campo ou evidência científica do WAIND.** Para dados reais, use a exportação privada do contrato, desative `MODO_TECNICO` e informe a pasta. O notebook e o dataset devem permanecer privados ao executar com dados reais.

Os cinco experimentos estão implementados. Texto e imagem usam extratores fixos e exigem habilitar `EXECUTAR_EMBEDDINGS`; quando desligados são explicitamente registrados como não executados. Não é criado um modelo fictício para substituir download ou dados ausentes.

In [ ]:
# Execute no Colab. Não coloca credenciais no notebook.
import sys, subprocess, importlib.util
requirements = ["numpy", "pandas", "scikit-learn==1.9.1", "Pillow", "joblib"]
if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *requirements])

In [ ]:
from pathlib import Path
import json, hashlib, re, math, platform, importlib.metadata, warnings
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import GroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor, NearestNeighbors
from sklearn.metrics import mean_absolute_error, accuracy_score, confusion_matrix
import joblib

MODO_TECNICO = True
EXECUTAR_EMBEDDINGS = False
SEED = 42
DATASET_DIR = Path("/content/dataset_privado")
RESULTS_DIR = Path("/content/waind_resultados_privados")
TEXT_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
TEXT_REVISION = None  # Fixe o commit dos pesos no estudo definitivo; registre-o nos resultados.
np.random.seed(SEED)

In [ ]:
QA_REGISTRY = {'schema_version': 1, 'notice': 'IDs exclusivamente técnicos. GPT07 reserva 32 UUIDs para testes locais. testes_producao_0_8: relatos de teste feitos no app publicado em 13/11/2026 (verificação da 0.8). testes_pre_coleta: relatos do pesquisador feitos para testar o app antes da coleta (30/10 a 13/11/2026). Excluir também qualquer ocorrência na coordenada sentinela (-80,0).', 'gpt05': ['128de2aa-edfa-489c-b048-e7b157e0ba31', '22a4fc49-488d-4a11-b5bb-60a04ac98a28', '4408bd78-7864-4a53-baa4-ae8fc11b0578', '4e0883fe-3295-4d16-9abf-0dd71aff8471', '8ec09065-06fc-41f1-9744-d8a2ac8920cb', 'a3cdd961-0b02-4632-aadc-a14fc30fd90a'], 'gpt06': ['c8e93016-9945-4244-86d8-7105f825da79', '04bdd533-5b30-4ffd-b44b-92e986abab28', '0c709360-20c0-4fc9-8bf2-ccdfb35cf5c3'], 'gpt07': ['70000000-0000-4000-8000-000000000001', '70000000-0000-4000-8000-000000000002', '70000000-0000-4000-8000-000000000003', '70000000-0000-4000-8000-000000000004', '70000000-0000-4000-8000-000000000005', '70000000-0000-4000-8000-000000000006', '70000000-0000-4000-8000-000000000007', '70000000-0000-4000-8000-000000000008', '70000000-0000-4000-8000-000000000009', '70000000-0000-4000-8000-000000000010', '70000000-0000-4000-8000-000000000011', '70000000-0000-4000-8000-000000000012', '70000000-0000-4000-8000-000000000013', '70000000-0000-4000-8000-000000000014', '70000000-0000-4000-8000-000000000015', '70000000-0000-4000-8000-000000000016', '70000000-0000-4000-8000-000000000017', '70000000-0000-4000-8000-000000000018', '70000000-0000-4000-8000-000000000019', '70000000-0000-4000-8000-000000000020', '70000000-0000-4000-8000-000000000021', '70000000-0000-4000-8000-000000000022', '70000000-0000-4000-8000-000000000023', '70000000-0000-4000-8000-000000000024', '70000000-0000-4000-8000-000000000025', '70000000-0000-4000-8000-000000000026', '70000000-0000-4000-8000-000000000027', '70000000-0000-4000-8000-000000000028', '70000000-0000-4000-8000-000000000029', '70000000-0000-4000-8000-000000000030', '70000000-0000-4000-8000-000000000031', '70000000-0000-4000-8000-000000000032'], 'testes_producao_0_8': ['1bc11e25-7c53-462c-acd4-d68b33795b4b', 'e64cc3c1-d2a8-4e0c-a93f-e089bd2d64c8', '1a4ac1c5-4a45-4d5a-9dc5-9bbb081765ec', 'cb5e0162-e184-41d8-a71f-bf8c76dc2457', '56a97a4f-f3ab-4788-88a4-7a4480e742dd'], 'testes_pre_coleta': ['a576bf82-d16a-4a3f-a968-baf4e84c13d8', 'b1bd6fbf-7efc-4007-bd95-aab43940c3cc', 'ba7c7bf6-854e-4959-94e9-b9c4b4d19a27', 'c5628820-ffd4-47e3-ad36-57e2e3d35d8b'], 'sentinel': {'latitude': -80, 'longitude': 0}}
QA_IDS = set(x for v in QA_REGISTRY.values() if isinstance(v, list) for x in v)
CATEGORIES = ["waste","container","sewer","ditch","organic","metabolic","chemical","other"]

## Lote exclusivamente técnico

O gerador escreve identificadores `sintetico-*`, fotos artificiais marcadas e origem `sintetico-pipeline`. Ele nunca é combinado automaticamente com dados de campo. Não altera Firebase, Drive ou GitHub.

In [ ]:
def mask_intensity(text):
    text = re.sub(r"\b(?:intensidade|nota|n[ií]vel|escala)\s*(?:(?:de|do|da|[ée]|igual\s+a)\s*)?[:=\-]?\s*[0-5](?:\s*(?:/|de)\s*5)?\b", "[INTENSIDADE]", str(text or ""), flags=re.I)
    text = re.sub(r"\b(?:muito\s+forte|muito\s+fraco|intermedi[aá]ri[oa]|moderad[oa]|forte|fraco)\b", "[INTENSIDADE]", text, flags=re.I)
    return re.sub(r"[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}", "[EMAIL]", text, flags=re.I)

def synthetic_batch(folder, n=60):
    folder.mkdir(parents=True, exist_ok=True); (folder/"fotos").mkdir(exist_ok=True)
    rng = np.random.default_rng(SEED); rows=[]
    for i in range(n):
        level=int(rng.integers(1,6)); has_photo=i%3!=0; has_text=i%4!=0; past=i%5==0
        day=f"2026-10-{i//10+1:02d}"; category=CATEGORIES[i%len(CATEGORIES)]
        text=(f"SIMULAÇÃO TÉCNICA de contexto urbano {i}. nota {level}" if has_text else "")
        visual={"caminho":None,"largura":None,"altura":None,"tem_foto":False}; sha=None; size=None
        if has_photo:
            rel=f"fotos/sintetico-{i:03d}.jpg"; image=Image.new("RGB",(24,16),(i*3%256,i*11%256,i*17%256)); image.save(folder/rel,quality=80)
            data=(folder/rel).read_bytes();sha=hashlib.sha256(data).hexdigest();size=len(data);visual={"caminho":rel,"largura":24,"altura":16,"tem_foto":True}
        weather={"temperatura_c":None if past else float(rng.uniform(15,30)),"umidade_pct":None if past else int(rng.integers(30,90)),"vento_kmh":None if past else float(rng.uniform(0,20)),"rajada_kmh":None if past else float(rng.uniform(5,40)),"vento_dir_sin":None if past else float(np.sin(rng.uniform(0,2*np.pi))),"vento_dir_cos":None if past else float(np.cos(rng.uniform(0,2*np.pi))),"condicao_codigo":None if past else 1}
        rows.append({"alvo":{"intensidade_humana":level},"entradas_estruturadas":{"categorias":[category],"multi_hot":{c:int(c==category) for c in CATEGORIES},"visivel_por_perto":[],"sem_foco_visivel":i%2==0,"data":day,"hora":f"{8+i%10:02d}:00:00","tipo_ocorrencia":"anterior" if past else "atual","latitude":-29.68+i/10000,"longitude":-53.81,"clima_valido_para_ocorrencia":not past,"clima":weather},"entradas_texto":{"outro_descricao":"","observacao":text,"texto_original":text,"texto_sem_indicio_direto":mask_intensity(text)},"entrada_visual":visual,"controle_coleta_e_divisao":{"id":f"sintetico-{i:03d}","origem_dataset":"sintetico-pipeline","consentimento_versao":"TECNICO-SEM-PARTICIPANTES","origem_localizacao":"simulada","tem_foto":has_photo,"tem_texto":has_text,"status_qualidade":"sintetico-pipeline","grupo_ocorrencia_ou_foto":f"grupo-tecnico-{i//2}","participante_pseudo_id":f"participante-tecnico-{i//10}","foto_sha256":sha,"foto_tamanho_bytes":size}})
    data="".join(json.dumps(r,ensure_ascii=False,allow_nan=False)+"\n" for r in rows).encode()
    (folder/"registros.jsonl").write_bytes(data)
    (folder/"manifesto.json").write_text(json.dumps({"schema_version":"waind-dataset-0.7","fonte":"sintetico-pipeline","sha256_registros":hashlib.sha256(data).hexdigest(),"qa_ids":sorted(QA_IDS)}),encoding="utf8")
    return rows

if MODO_TECNICO:
    DATASET_DIR = RESULTS_DIR/"lote_sintetico_pipeline"
    synthetic_batch(DATASET_DIR)
    print("VERIFICAÇÃO TÉCNICA. Nenhum dado de campo e nenhuma alegação de desempenho real.")

## Integridade, ausências e bloqueio de entradas indevidas

O manifesto e os arquivos são verificados antes de qualquer experimento. O alvo e os controles nunca entram em X. Texto original e versão mascarada são alternativas declaradas, não colunas acrescentadas automaticamente. Dados meteorológicos atuais de uma ocorrência passada são anulados. Fotos ausentes permanecem ausentes.

In [ ]:
FORBIDDEN = {"nome","name","email","e-mail","autor_uid","uid","provedor_auth","intensidade_humana","intensidade_odor","rotulos_ml","intensidade_nivel","intensidade_estimada","alvo"}
def assert_inputs(value):
    if isinstance(value,dict):
        for key,item in value.items():
            assert key.lower() not in FORBIDDEN, f"Entrada proibida: {key}"
            assert_inputs(item)
    elif isinstance(value,list):
        for item in value: assert_inputs(item)

manifest=json.loads((DATASET_DIR/"manifesto.json").read_text(encoding="utf8"))
raw=(DATASET_DIR/"registros.jsonl").read_bytes()
assert manifest["schema_version"]=="waind-dataset-0.7"
assert hashlib.sha256(raw).hexdigest()==manifest["sha256_registros"], "Manifesto divergente"
qa_ids=QA_IDS|set(manifest.get("qa_ids",[]));rows=[];ignored=[]
for line in raw.decode("utf8").splitlines():
    r=json.loads(line);e=r["entradas_estruturadas"];c=r["controle_coleta_e_divisao"];v=r["entrada_visual"];t=r["entradas_texto"]
    if c["id"] in qa_ids or (e["latitude"]==-80 and e["longitude"]==0): ignored.append(c["id"]);continue
    assert bool(c.get("origem_dataset")=="sintetico-pipeline")==MODO_TECNICO, "Dados reais e sintéticos não podem ser misturados"
    assert_inputs({"estruturadas":e,"texto":t,"visual":v})
    y=r["alvo"]["intensidade_humana"];assert type(y) is int and 1<=y<=5
    assert e["tipo_ocorrencia"] in ["atual","anterior"]
    assert set(e["multi_hot"])==set(CATEGORIES)
    assert all(e["multi_hot"][k]==int(k in e["categorias"]) for k in CATEGORIES)
    # Ocorrência anterior só mantém clima quando o exportador marcou clima horário histórico (0.8).
    if e["tipo_ocorrencia"]=="anterior" and not e.get("clima_valido_para_ocorrencia"):
        e["clima_valido_para_ocorrencia"]=False;e["clima"]={k:None for k in e["clima"]}
    assert c["tem_foto"]==v["tem_foto"]
    if v["tem_foto"]:
        picture=(DATASET_DIR/v["caminho"]).resolve();assert picture.is_relative_to(DATASET_DIR.resolve()), "Caminho de foto inválido"
        data=picture.read_bytes();assert len(data)==c["foto_tamanho_bytes"] and hashlib.sha256(data).hexdigest()==c["foto_sha256"]
        with Image.open(picture) as im: assert im.size==(v["largura"],v["altura"]);im.verify()
    else: assert v["caminho"] is None
    assert c["tem_texto"]==bool(t["texto_original"].strip())
    assert c.get("grupo_ocorrencia_ou_foto"), "Grupo obrigatório"
    rows.append(r)
assert rows, "Não há registros elegíveis para avaliar"
print({"registros_elegiveis":len(rows),"qa_excluidos":len(ignored),"modo":"tecnico" if MODO_TECNICO else "campo"})

## Entradas explícitas e divisões sem sobreposição

Cada avaliação usa grupos de ocorrência/foto. Para avaliar por dia ou participante, componentes conectados unem também esses eixos; assim a mesma foto não reaparece no outro lado. Se isso formar poucos grupos, o experimento é interrompido ou marcado como inviável. Não se troca silenciosamente por divisão aleatória.

In [ ]:
def structured_features(records):
    items=[]
    tags=["Lixo acumulado","Contentor próximo","Bueiro / ralo","Água parada","Vala aberta","Restos de alimentos","Animais por perto","Nada aparente"]
    for r in records:
        e=r["entradas_estruturadas"];c=r["controle_coleta_e_divisao"];hour=int(e["hora"][:2]);weekday=pd.Timestamp(e["data"]).dayofweek
        d={"cat_"+k:float(e["multi_hot"][k]) for k in CATEGORIES}
        d.update({"tag_"+str(i):float(t in e["visivel_por_perto"]) for i,t in enumerate(tags)})
        d.update(latitude=e["latitude"],longitude=e["longitude"],hora_sin=math.sin(2*math.pi*hour/24),hora_cos=math.cos(2*math.pi*hour/24),dia_semana=weekday,sem_foco=float(e["sem_foco_visivel"]),ocorrencia_anterior=float(e["tipo_ocorrencia"]=="anterior"),clima_valido=float(e["clima_valido_para_ocorrencia"]),tem_foto=float(c["tem_foto"]),tem_texto=float(c["tem_texto"]))
        d.update({"clima_"+k:(float(v) if v is not None else np.nan) for k,v in e["clima"].items()})
        items.append(d)
    frame=pd.DataFrame(items);assert not(set(frame.columns)&FORBIDDEN);return frame.astype(float).to_numpy(),list(frame.columns)

X_struct,feature_names=structured_features(rows)
y=np.array([r["alvo"]["intensidade_humana"] for r in rows],dtype=int)
photo_mask=np.array([r["entrada_visual"]["tem_foto"] for r in rows]);text_mask=np.array([r["controle_coleta_e_divisao"]["tem_texto"] for r in rows])
def split_groups(axis):
    parent=list(range(len(rows)))
    def find(i):
        while parent[i]!=i: parent[i]=parent[parent[i]];i=parent[i]
        return i
    def join(i,j): parent[find(i)]=find(j)
    seen={}
    for i,r in enumerate(rows):
        c=r["controle_coleta_e_divisao"];keys=[("ocorrencia",c["grupo_ocorrencia_ou_foto"])]
        if c.get("foto_sha256"):keys.append(("foto",c["foto_sha256"]))
        if axis=="dia":keys.append(("dia",r["entradas_estruturadas"]["data"]))
        if axis=="participante":
            assert c.get("participante_pseudo_id"), "Não há autoria pseudonimizada para avaliar novos participantes"
            keys.append(("participante",c["participante_pseudo_id"]))
        for key in keys:
            if key in seen:join(i,seen[key])
            else:seen[key]=i
    return np.array([find(i) for i in range(len(rows))])

SPLITS={}
for axis in ["ocorrencia","dia","participante"]:
    try:
        groups=split_groups(axis);count=len(np.unique(groups));assert count>=2,"Menos de dois grupos independentes"
        SPLITS[axis]=list(GroupKFold(n_splits=min(3,count)).split(X_struct,y,groups))
    except AssertionError as error: print(f"Divisão {axis} não executada: {error}")
assert SPLITS, "Não há grupos suficientes para avaliação"

## Extratores fixos de texto e imagem

Ative `EXECUTAR_EMBEDDINGS` para executar os experimentos 3–5. O download dos pesos exige internet. A base MobileNetV2 fica congelada; não se treina a rede visual do zero. Registros sem foto não são enviados ao extrator. Os vetores ausentes ficam como NaN; a imputação numérica posterior é ajustada no treino e inclui indicadores de ausência, sem fabricar arquivos de imagem.

In [ ]:
X_text_original=X_text_masked=X_image=None
if EXECUTAR_EMBEDDINGS:
    subprocess.check_call([sys.executable,"-m","pip","install","-q","sentence-transformers>=3,<6","tensorflow>=2.16,<3"])
    from sentence_transformers import SentenceTransformer
    import tensorflow as tf
    tf.keras.utils.set_random_seed(SEED)
    encoder=SentenceTransformer(TEXT_MODEL,revision=TEXT_REVISION)
    def text_embeddings(field):
        dim=encoder.get_sentence_embedding_dimension();out=np.full((len(rows),dim),np.nan)
        selected=[i for i,r in enumerate(rows) if r["entradas_texto"][field].strip()]
        if selected:out[selected]=encoder.encode([rows[i]["entradas_texto"][field] for i in selected],normalize_embeddings=True,show_progress_bar=True)
        return out
    X_text_original=text_embeddings("texto_original");X_text_masked=text_embeddings("texto_sem_indicio_direto")
    visual=tf.keras.applications.MobileNetV2(weights="imagenet",include_top=False,pooling="avg",input_shape=(224,224,3));visual.trainable=False
    X_image=np.full((len(rows),1280),np.nan)
    for i,r in enumerate(rows):
        if not r["entrada_visual"]["tem_foto"]:continue
        with Image.open(DATASET_DIR/r["entrada_visual"]["caminho"]) as im:array=np.asarray(im.convert("RGB").resize((224,224)),dtype=np.float32)
        vector=visual.predict(tf.keras.applications.mobilenet_v2.preprocess_input(array[None]),verbose=0)[0]
        X_image[i]=vector/max(np.linalg.norm(vector),1e-12)
else:
    print("Experimentos de texto/imagem/completo: não executados; habilite EXECUTAR_EMBEDDINGS.")

In [ ]:
def preprocessing():return make_pipeline(SimpleImputer(strategy="constant",fill_value=0,add_indicator=True,keep_empty_features=True),StandardScaler())
experiments={
 "1_referencia_mediana":(X_struct,lambda:DummyRegressor(strategy="median")),
 "2_arvore_estruturada":(X_struct,lambda:make_pipeline(preprocessing(),DecisionTreeRegressor(max_depth=5,min_samples_leaf=3,random_state=SEED))),
 "2_floresta_estruturada":(X_struct,lambda:make_pipeline(preprocessing(),RandomForestRegressor(n_estimators=100,min_samples_leaf=3,random_state=SEED,n_jobs=1)))
}
if EXECUTAR_EMBEDDINGS:
    for label,embedding in [("original",X_text_original),("mascarado",X_text_masked)]:
        experiments["3_texto_"+label]=(np.c_[X_struct,embedding],lambda:make_pipeline(preprocessing(),Ridge(alpha=10)))
    experiments["4_contexto_imagem"]=(np.c_[X_struct,X_image],lambda:make_pipeline(preprocessing(),Ridge(alpha=10)))
    experiments["4_vizinhos_contexto_imagem"]=(np.c_[X_struct,X_image],lambda:make_pipeline(preprocessing(),KNeighborsRegressor(n_neighbors=3,weights="distance")))
    experiments["5_completo_mascarado"]=(np.c_[X_struct,X_text_masked,X_image],lambda:make_pipeline(preprocessing(),Ridge(alpha=10)))

def metrics(actual,predicted):
    return {"n":len(actual),"mae_niveis":float(mean_absolute_error(actual,predicted)),"acerto_exato":float(accuracy_score(actual,predicted)),"matriz_confusao":confusion_matrix(actual,predicted,labels=[1,2,3,4,5]).tolist()}

results=[];predictions=[]
for axis,splits in SPLITS.items():
    for name,(X,factory) in experiments.items():
        output=np.full(len(rows),np.nan);abstain=np.zeros(len(rows),dtype=bool)
        for fold,(train,test) in enumerate(splits):
            model=factory();model.fit(X[train],y[train]);raw=model.predict(X[test]);output[test]=np.clip(np.rint(raw),1,5)
            if name.startswith("5_"):
                # Regra experimental de suporte. Limiar vem SOMENTE do treino; não é confiança calibrada.
                prep=preprocessing();z=prep.fit_transform(X[train]);zt=prep.transform(X[test]);nn=NearestNeighbors(n_neighbors=2).fit(z)
                threshold=float(np.quantile(nn.kneighbors(z)[0][:,1],0.95));dist=nn.kneighbors(zt,n_neighbors=1)[0][:,0]
                abstain[test]=dist>threshold
        assert np.isfinite(output).all();output=output.astype(int)
        summary={"experimento":name,"divisao":axis,"origem":"sintetico-pipeline" if MODO_TECNICO else "campo",**metrics(y,output)}
        summary["por_nivel"]={str(level):metrics(y[y==level],output[y==level]) if np.any(y==level) else {"n":0,"sem_dados":True} for level in range(1,6)}
        summary["subconjuntos"]={}
        for label,mask in [("com_foto",photo_mask),("sem_foto",~photo_mask),("com_texto",text_mask),("sem_texto",~text_mask)]:
            summary["subconjuntos"][label]=metrics(y[mask],output[mask]) if mask.any() else {"n":0,"sem_dados":True}
        if name.startswith("5_"):
            covered=~abstain;summary["abstencao"]={"cobertura":float(covered.mean()),"n_abstencoes":int(abstain.sum()),"metricas_cobertos":metrics(y[covered],output[covered]) if covered.any() else {"n":0},"aviso":"Limiar experimental; validar antes de integração"}
        results.append(summary)
        predictions.extend({"id":rows[i]["controle_coleta_e_divisao"]["id"],"experimento":name,"divisao":axis,"intensidade_humana":int(y[i]),"intensidade_estimada":None if abstain[i] else int(output[i]),"abstencao":bool(abstain[i])} for i in range(len(rows)))

table=pd.DataFrame([{k:r[k] for k in ["experimento","divisao","origem","n","mae_niveis","acerto_exato"]} for r in results])
print("RESULTADOS EXCLUSIVAMENTE TÉCNICOS, SEM EVIDÊNCIA DE CAMPO" if MODO_TECNICO else "Avaliação exploratória em dados de campo; não usar o teste para escolher hiperparâmetros")
print(table.to_string(index=False))

## Salvar evidências privadas e planejar o teste final

As comparações acima são validações agrupadas de configurações previamente declaradas. Escolhas posteriores exigem uma nova rodada de validação; um conjunto final independente deve permanecer reservado antes de otimizar hiperparâmetros. Não use essas mesmas métricas repetidamente como teste final.

A integração futura exigirá modelo versionado, cobertura/abstenção avaliadas e ausência de alterações na intensidade humana. Nenhuma escrita no Firebase é realizada por este notebook.

In [ ]:
RESULTS_DIR.mkdir(parents=True,exist_ok=True)
environment={"python":platform.python_version(),"seed":SEED,"modo_tecnico":MODO_TECNICO,"embeddings_executados":EXECUTAR_EMBEDDINGS,"text_model":TEXT_MODEL if EXECUTAR_EMBEDDINGS else None,"text_revision":TEXT_REVISION,"dataset_sha256":manifest["sha256_registros"],"pacotes":{}}
for package in ["numpy","pandas","scikit-learn","Pillow","sentence-transformers","tensorflow"]:
    try:environment["pacotes"][package]=importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:pass
(RESULTS_DIR/"ambiente.json").write_text(json.dumps(environment,indent=2),encoding="utf8")
(RESULTS_DIR/"metricas.json").write_text(json.dumps(results,ensure_ascii=False,indent=2,allow_nan=False),encoding="utf8")
(RESULTS_DIR/"predicoes_avaliacao.json").write_text(json.dumps(predictions,ensure_ascii=False,indent=2,allow_nan=False),encoding="utf8")
(RESULTS_DIR/"divisoes.json").write_text(json.dumps({axis:[{"treino":[rows[i]["controle_coleta_e_divisao"]["id"] for i in train],"teste":[rows[i]["controle_coleta_e_divisao"]["id"] for i in test]} for train,test in splits] for axis,splits in SPLITS.items()},indent=2),encoding="utf8")
print("Arquivos privados gravados em",RESULTS_DIR)

## Fontes e limites

- [Contrato do dataset](CONTRATO-DATASET-PESQUISA.md).
- [scikit-learn: vazamento de dados](https://scikit-learn.org/stable/common_pitfalls.html).
- [GroupKFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupKFold.html).
- [TensorFlow: extração visual por transferência de aprendizado](https://www.tensorflow.org/tutorials/images/transfer_learning).
- [Sentence Transformers: modelos multilíngues](https://www.sbert.net/docs/sentence_transformer/pretrained_models.html).

O local pode virar um atalho de memorização; a avaliação por locais novos ainda deve ser desenhada para o plano de campo. Agrupar ocorrências por proximidade não comprova que tenham a mesma origem. Fotografias não medem cheiro. Pseudônimos, texto, imagens e coordenadas continuam sendo informação privada.